In [1]:
# Cell 1: Dataset Ingestion & Analytics

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# 1. Load the dataset uploaded to your Colab directory
df = pd.read_csv('customer_support_text_classification.csv')

print("=== Task 1: Dataset Understanding ===")
print(f"Total Number of Records: {len(df)}")
print(f"\nUnique Target Labels: {df['sentiment_label'].unique()}")

print("\n--- Sample Text Records ---")
for idx, row in df.head(3).iterrows():
    print(f"Sentiment [{row['sentiment_label'].upper()}]: '{row['customer_message']}'")

# Calculate average text statistics
df['calculated_word_count'] = df['customer_message'].apply(lambda x: len(str(x).split()))
print(f"\nAverage Text Word Count: {df['calculated_word_count'].mean():.2f} words")

print("\n--- Class Distribution ---")
class_dist = df['sentiment_label'].value_counts()
for cls, val in class_dist.items():
    pct = (val / len(df)) * 100
    print(f" Class '{cls}': {val} records ({pct:.2f}%)")

=== Task 1: Dataset Understanding ===
Total Number of Records: 1500

Unique Target Labels: ['neutral' 'positive' 'negative']

--- Sample Text Records ---
Sentiment [NEUTRAL]: 'I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.'
Sentiment [NEUTRAL]: 'I need information about the payment process.'
Sentiment [POSITIVE]: 'The refund process was fast and convenient. I appreciate the quick response.'

Average Text Word Count: 12.72 words

--- Class Distribution ---
 Class 'neutral': 524 records (34.93%)
 Class 'negative': 497 records (33.13%)
 Class 'positive': 479 records (31.93%)


In [2]:
# Cell 2: Advanced Cleaning & Normalization

print("=== Task 2: Text Preprocessing ===")

# Standard list of English stopwords for cleaning
STOPWORDS = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself",
    "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this",
    "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and",
    "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with",
    "about", "against", "between", "into", "through", "during", "before", "after", "above",
    "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again",
    "further", "then", "once"
])

def clean_text(text):
    text = str(text).lower() # Lowercasing
    text = re.sub(r'ticket number is \d+', '', text) # Remove ticket sentence noise
    text = re.sub(r'\b\d+\b', '', text) # Remove leftover digits
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation marks
    tokens = text.split() # Tokenize
    cleaned_tokens = [w for w in tokens if w not in STOPWORDS] # Stopword suppression
    return " ".join(cleaned_tokens)

df['cleaned_message'] = df['customer_message'].apply(clean_text)
print("\n--- Preprocessing Evaluation ---")
print(f"Original : {df['customer_message'].iloc[0]}")
print(f"Processed: {df['cleaned_message'].iloc[0]}")

=== Task 2: Text Preprocessing ===

--- Preprocessing Evaluation ---
Original : I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Processed: need information payment process please respond soon possible


In [3]:
# Cell 3: Classical Vectorization & Baseline Model

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("=== Task 3 & 4: Text Vectorization & Baseline Model ===")

X = df['cleaned_message']
y = df['sentiment_label']

# Split dataset into training sets (80%) and validation sets (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train classical Logistic Regression model
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, y_train)

# Evaluate predictions
y_pred = baseline_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nClassical Baseline Model Accuracy: {accuracy*100:.2f}%")
print("\n--- Comprehensive Performance Metrics ---")
print(classification_report(y_test, y_pred))

=== Task 3 & 4: Text Vectorization & Baseline Model ===

Classical Baseline Model Accuracy: 100.00%

--- Comprehensive Performance Metrics ---
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [4]:
# Cell 4: LSTM Sequence Deep Learning Model

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

print("=== Task 5: Sequence Model Implementation ===")

# Map string text categories to simple logic numbers
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
y_train_encoded = np.array([label_mapping[label] for label in y_train])
y_test_encoded = np.array([label_mapping[label] for label in y_test])

MAX_VOCAB = 1000
MAX_SEQUENCE_LENGTH = 25

# Setup tokenizer pipeline
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_SEQUENCE_LENGTH, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_SEQUENCE_LENGTH, padding='post')

# Define Deep Learning Sequential Blueprint
model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=32, input_length=MAX_SEQUENCE_LENGTH),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax') # Multi-class probability calculator
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train the LSTM deep sequence model
print("\n--- Training Sequence Model ---")
history = model.fit(X_train_seq, y_train_encoded, epochs=5, batch_size=32, validation_data=(X_test_seq, y_test_encoded), verbose=1)

=== Task 5: Sequence Model Implementation ===


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


--- Training Sequence Model ---
Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.3342 - loss: 1.0990 - val_accuracy: 0.3500 - val_loss: 1.0981
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.4542 - loss: 1.0206 - val_accuracy: 0.6800 - val_loss: 0.5996
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.6733 - loss: 0.5312 - val_accuracy: 0.6700 - val_loss: 0.4544
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6758 - loss: 0.4653 - val_accuracy: 0.6700 - val_loss: 0.4492
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.7242 - loss: 0.4517 - val_accuracy: 0.6967 - val_loss: 0.4106


In [5]:
# Cell 5: Save and Export Metric Output Files

import os

# Create local results folder structure inside Colab
os.makedirs('results', exist_ok=True)

# Generate and export metrics table file
report_dict = classification_report(y_test, y_pred, output_dict=True)
metrics_df = pd.DataFrame(report_dict).transpose()
metrics_df.to_csv('results/model_evaluation.csv', index=True)
print("\n[Saved] Metrics exported to 'results/model_evaluation.csv'")

# Generate sample predictions file
sample_messages = [
    "This application is broken and crashes whenever I press checkout.",
    "The shipment arrived on time and the tracking details were excellent.",
    "Can you please explain how to upgrade to a premium account subscription?"
]
cleaned_samples = [clean_text(msg) for msg in sample_messages]
sample_seqs = pad_sequences(tokenizer.texts_to_sequences(cleaned_samples), maxlen=MAX_SEQUENCE_LENGTH, padding='post')

predictions = model.predict(sample_seqs)
predicted_classes = np.argmax(predictions, axis=1)
reverse_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}

with open('results/sample_predictions.txt', 'w') as f:
    f.write("=== Execution Sample Predictions ===\n\n")
    for raw, clean, pred_idx in zip(sample_messages, cleaned_samples, predicted_classes):
        out_line = f"Message: {raw}\nPredicted Sentiment: {reverse_mapping[pred_idx].upper()}\n\n"
        f.write(out_line)
        print(out_line.strip())

print("[Saved] Production predictions exported to 'results/sample_predictions.txt'")


[Saved] Metrics exported to 'results/model_evaluation.csv'
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 666ms/step
Message: This application is broken and crashes whenever I press checkout.
Predicted Sentiment: NEGATIVE
Message: The shipment arrived on time and the tracking details were excellent.
Predicted Sentiment: NEGATIVE
Message: Can you please explain how to upgrade to a premium account subscription?
Predicted Sentiment: NEUTRAL
[Saved] Production predictions exported to 'results/sample_predictions.txt'
